# Predictive Modeling and Risk Scoring for Bank Customer Churn

This notebook builds an end-to-end churn intelligence system using the supplied `European_Bank.csv` dataset.

**Goal:** predict `Exited`, produce a churn probability/risk score, compare ML models, explain drivers, and prepare outputs for a Streamlit application.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier
import joblib

RANDOM_STATE = 42


In [ ]:
df = pd.read_csv("European_Bank.csv")
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)


## 1. Data Quality and Exploratory Analysis

The supplied dataset contains 10,000 records and 14 columns. The target is `Exited`, where 1 means churned and 0 means retained.


In [ ]:
print("Missing values:")
display(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())
print("Target distribution:")
display(df["Exited"].value_counts())
display(df["Exited"].value_counts(normalize=True).rename("proportion"))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Exited")
plt.title("Customer Churn Distribution")
plt.show()


In [ ]:
# Business-oriented EDA
fig, axes = plt.subplots(1, 3, figsize=(16,4))

sns.boxplot(data=df, x="Exited", y="Age", ax=axes[0])
axes[0].set_title("Age vs Churn")

sns.boxplot(data=df, x="Exited", y="CreditScore", ax=axes[1])
axes[1].set_title("Credit Score vs Churn")

sns.barplot(data=df, x="Geography", y="Exited", ax=axes[2])
axes[2].set_title("Churn Rate by Geography")

plt.tight_layout()
plt.show()

for col in ["Gender","IsActiveMember","HasCrCard","NumOfProducts","Tenure"]:
    print("\n", col)
    display(df.groupby(col)["Exited"].agg(["count","mean"]).rename(columns={"mean":"churn_rate"}))


## 2. Feature Engineering

Removed:
- `CustomerId` and `Surname`: identifiers, not predictive business features.
- `Year`: treated as metadata rather than a customer-behavior feature because it is constant in the supplied file.

Created:
- `BalanceSalaryRatio`
- `ProductDensity`
- `EngagementProduct`
- `AgeTenureInteraction`


In [ ]:
data = df.copy()

data["BalanceSalaryRatio"] = data["Balance"] / (data["EstimatedSalary"] + 1)
data["ProductDensity"] = data["NumOfProducts"] / (data["Tenure"] + 1)
data["EngagementProduct"] = data["IsActiveMember"] * data["NumOfProducts"]
data["AgeTenureInteraction"] = data["Age"] * (data["Tenure"] + 1)

X = data.drop(columns=["Exited","CustomerId","Surname","Year"])
y = data["Exited"]

categorical_features = ["Geography","Gender"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]), categorical_features)
])


## 3. Model Development and Evaluation

Models:
1. Logistic Regression — interpretable baseline.
2. Random Forest — nonlinear tree ensemble.
3. Gradient Boosting — strong tabular-data benchmark.
4. XGBoost — advanced gradient boosting model.

Because churn is imbalanced, do not rely on accuracy alone. Precision, recall, F1 and ROC-AUC are reported.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.04,
        subsample=0.85, colsample_bytree=0.85,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=4
    )
}

pipelines = {}
results = []

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    prob = pipe.predict_proba(X_test)[:,1]
    pred = (prob >= 0.50).astype(int)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })
    pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.style.format({
    "Accuracy":"{:.3f}", "Precision":"{:.3f}", "Recall":"{:.3f}",
    "F1":"{:.3f}", "ROC-AUC":"{:.3f}"
}))


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_model_name]
test_prob = best_model.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, test_prob)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"{best_model_name} AUC={roc_auc_score(y_test,test_prob):.3f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Threshold analysis: choose the threshold with maximum F1 on the test set for demonstration.
thresholds = np.arange(0.20, 0.71, 0.01)
threshold_rows = []
for t in thresholds:
    pred = (test_prob >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "precision": precision_score(y_test,pred,zero_division=0),
        "recall": recall_score(y_test,pred,zero_division=0),
        "f1": f1_score(y_test,pred,zero_division=0),
        "accuracy": accuracy_score(y_test,pred)
    })
threshold_df = pd.DataFrame(threshold_rows)
best_threshold = float(threshold_df.loc[threshold_df["f1"].idxmax(),"threshold"])
print("Best F1 threshold:", best_threshold)
display(threshold_df.sort_values("f1", ascending=False).head(10))


## 4. Risk Scoring

Suggested business bands:
- **Low:** probability < 0.30
- **Medium:** 0.30–0.60
- **High:** 0.60–0.80
- **Critical:** >= 0.80

These are operational bands, not probabilities of certainty. A bank should recalibrate them against campaign capacity, intervention cost and retention value.


In [ ]:
risk_output = X_test.copy()
risk_output["ActualExited"] = y_test.values
risk_output["ChurnProbability"] = test_prob
risk_output["RiskBand"] = pd.cut(
    risk_output["ChurnProbability"],
    bins=[-np.inf,0.30,0.60,0.80,np.inf],
    labels=["Low","Medium","High","Critical"]
)
display(risk_output.head())

plt.figure(figsize=(8,4))
sns.histplot(test_prob, bins=20, kde=True)
plt.xlabel("Predicted churn probability")
plt.title("Predicted Churn Probability Distribution")
plt.show()


## 5. Feature Importance

For tree models, permutation importance is a model-agnostic option and is used here to avoid relying on model-specific internal representations after preprocessing.


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_test, y_test, scoring="roc_auc",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

plt.figure(figsize=(9,6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top Feature Importance (Permutation, ROC-AUC)")
plt.xlabel("Mean importance")
plt.show()

display(importance.head(15).to_frame("importance"))


## 6. Save the Model

The saved pipeline contains preprocessing plus the trained model, so the Streamlit app can accept raw customer inputs.


In [ ]:
joblib.dump({
    "pipeline": best_model,
    "threshold": best_threshold,
    "risk_bands": {"Low":0.30, "Medium":0.60, "High":0.80},
    "model_name": best_model_name
}, "bank_churn_model.joblib")

print("Saved: bank_churn_model.joblib")
